# Robots Dynamic: автономный pipeline

Этот notebook выполняет полный сценарий:

1. загрузка фактических данных;
2. обучение one-step GNN;
3. обучение Macrostate KMeans на fact-data;
4. multistep evaluation на fact-data;
5. генерация синтетических начальных условий;
6. synthetic rollout на 600 шагов;
7. анализ формирования мицеллы.

Единственный обязательный внешний файл: `data/processed_robots_data.parquet`. Веса GNN обучаются и сохраняются внутри этой папки.

In [1]:
from pathlib import Path

from config import ResearchConfig
from experiment_runner import ResearchPipeline
from initial_state_generator import InitialConditionSpec, InitialStateGenerator

# Центральная конфигурация
config = ResearchConfig(
    data_path="data/processed_robots_data.parquet",
    model_path="models/gnn_model_weights.pth",
    output_dir="results/research",

    # GNN
    hidden_dim=128,
    gnn_layers=2,
    dropout=0.1,
    k_neighbors=5,

    # Обучение one-step
    train_one_step=True,
    skip_training_if_model_exists=True,  # поставьте False или force=True ниже для переобучения
    batch_size=512,
    num_epochs=50,
    learning_rate=1e-3,
    weight_decay=1e-4,
    early_stopping_patience=10,
    # standardize_gnn=False,
    train_sample_ratio=0.35,  # доля fact-файлов для обучения: 1.0 = все, 0.5 = 50%
    max_train_samples_per_file=None,  # можно поставить число для ускоренного debug

    # Multistep / synthetic
    start_step=3,
    synthetic_total_steps=600,
    n_robots=50,
    dt=0.1,

    # Macrostate
    n_clusters=4,
    robot_size=60.0,
    random_state=42,

    # Видео/анимации: создаются отдельными ячейками после обучения/evaluation
    save_animations=True,
    auto_save_animations_during_pipeline=False,  # видео НЕ создаётся внутри train/evaluate
    animation_fps=10,
    animation_max_frames=300,  # None или -1 в CLI для всех кадров

    # Демонстрационный эксперимент для графиков/видео.
    # Логика как в GitHub main.ipynb: выбрать конкретный test-файл, например test_files[2].
    demo_split="test",
    demo_file_index=2,
    demo_file_name=None,  # можно явно указать: "01_69_[45_bots_PWM_2_ex_108].pickle"
)

config.ensure_dirs()
pipeline = ResearchPipeline(config)
print(config)

ResearchConfig(data_path='data/processed_robots_data.parquet', model_path='models/gnn_model_weights.pth', output_dir='results/research', model_save_path='models/gnn_model_weights.pth', results_save_path='results/gnn_predictions.parquet', k_neighbors=5, hidden_dim=128, edge_dim=32, gnn_layers=2, gnn_type='GIN', dropout=0.1, train_one_step=True, skip_training_if_model_exists=True, batch_size=512, num_epochs=50, learning_rate=0.001, weight_decay=0.0001, early_stopping_patience=10, show_progress=True, log_interval=10, grad_clip=1.0, lr_patience=5, lr_factor=0.5, train_sample_ratio=0.35, save_one_step_predictions_after_training=True, test_size=0.2, val_size=0.1, num_workers=0, data_device='cpu', one_step_min_t=2, max_train_samples_per_file=None, max_val_samples_per_file=None, max_test_samples_per_file=None, dt=0.1, start_step=3, synthetic_total_steps=600, n_robots=50, robot_size=60.0, dish_diameter=1000.0, min_center_distance=None, speed_min=10.0, speed_max=35.0, n_speed_samples=3, n_cluste

## 1. Загрузка fact-data

Данные должны лежать в `data/processed_robots_data.parquet`.

In [2]:
experiments = pipeline.load_fact_data()
print(f"Loaded experiments: {len(experiments)}")
first_name = next(iter(experiments))
print("First experiment:", first_name, experiments[first_name]["tensor"].shape)

🚀 Векторизованная загрузка данных...
📊 Загружено 307 экспериментов на cpu
Loaded experiments: 307
First experiment: 00_61_[45_bots_PWM_2_ex_175].pickle torch.Size([609, 45, 3])


## 2. Обучение one-step GNN

Модель обучается внутри автономной папки и сохраняется в `models/gnn_model_weights.pth`.

Если файл уже существует и `skip_training_if_model_exists=True`, обучение будет пропущено. Для принудительного переобучения поставьте `force=True`.

In [4]:
one_step_metrics = pipeline.train_one_step_model(experiments, force=False)
one_step_metrics

Using train_sample_ratio=0.350: 107/307 fact files
One-step split:
  train files: 74
  val files:   11
  test files:  22
🚀 Подготовка данных для GNN...
🚀 Запуск оптимизированного GraphDataset с полной векторизацией...
📊 Всего графов для создания: 43451


🔧 Векторизованное создание графов:   0%|          | 0/43451 [00:00<?, ?it/s]

✅ Создано 43451 графов (оптимизированная векторизация)
🚀 Запуск оптимизированного GraphDataset с полной векторизацией...
📊 Всего графов для создания: 6766


🔧 Векторизованное создание графов:   0%|          | 0/6766 [00:00<?, ?it/s]

✅ Создано 6766 графов (оптимизированная векторизация)
🚀 Запуск оптимизированного GraphDataset с полной векторизацией...
📊 Всего графов для создания: 13095


...

Epoch 50/50:   0%|          | 0/85 [00:00<?, ?it/s]

⏱ Обучение заняло: 2.1 минут
🔍 Тестирование...


Тестирование:   0%|          | 0/26 [00:00<?, ?it/s]

📊 Результаты Тестирование:
  gnn_rmse_delta_x: 2.5679
  gnn_rmse_delta_y: 2.4428
  gnn_rmse_delta_angle: 4.0094
  gnn_mape_delta_x: 1863.7413
  gnn_mape_delta_y: 1844.5835
  gnn_mape_delta_angle: 2846.2910
  total_predictions: 589275.0000
🔮 Прогнозирование для всех данных...
⏱ Inference занял: 12.8 минут
💾 Результаты GNN сохранены: results/gnn_predictions.parquet


{'gnn_train_time_minutes': 2.0774338285128278,
 'final_train_loss': 0.5605777431936825,
 'best_val_loss': 0.577426437820707,
 'gnn_rmse_delta_x': 2.567873954772949,
 'gnn_rmse_delta_y': 2.442768096923828,
 'gnn_rmse_delta_angle': 4.009427547454834,
 'gnn_mape_delta_x': 1863.7413330078125,
 'gnn_mape_delta_y': 1844.58349609375,
 'gnn_mape_delta_angle': 2846.291015625,
 'total_predictions': 589275,
 'train_sample_ratio': 0.35,
 'device': 'cuda',
 'model_path': 'models/gnn_model_weights.pth',
 'results_path': 'results/gnn_predictions.parquet'}

## 2.1. Видео one-step после обучения

Эта ячейка не переобучает модель. Она читает сохранённый `results/gnn_predictions.parquet` и создаёт видео.


In [3]:
config.demo_split = "test"
config.demo_file_index = 2

demo_file = pipeline.select_demo_file(experiments)
# demo_file

one_step_video_path = pipeline.save_one_step_video(file_name=demo_file)
one_step_video_path


Using train_sample_ratio=0.350: 107/307 fact files
Selected demo experiment: 02_63_[45_bots_PWM_2_ex_158].pickle (test_files[2])
Animation created for 300 frames and 45 robots: results\research\videos\one_step_real_vs_predicted_02_63__45_bots_PWM_2_ex_158__pickle.mp4


WindowsPath('results/research/videos/one_step_real_vs_predicted_02_63__45_bots_PWM_2_ex_158__pickle.mp4')

## 3. Обучение Macrostate KMeans на фактических данных

Кластеры/состояния системы берутся из fact-data. Синтетические траектории дальше только классифицируются этим KMeans.

In [3]:
macro_classifier = pipeline.fit_macrostate_classifier(experiments)
macro_classifier.interpretation_df

Extracting fact-data macrostate features...
 Cluster  Samples                        State_Type    PO  Angular_Vel  Coord_Num  Mean_Dist  Vel_Disp
       0    72356       Regular Structure (Мицелла) 0.135        0.044        4.2      368.4     0.677
       1    11420             Chaotic Motion (Хаос) 0.133        0.072        3.5      396.6     0.808
       2    74520 Sparse Distribution (Разреженное) 0.133        0.042        3.6      413.1     0.666
       3    23598 Sparse Distribution (Разреженное) 0.135       -0.023        3.9      390.9     0.740


,Cluster,Samples,State_Type,PO,Angular_Vel,Coord_Num,Mean_Dist,Vel_Disp
0,0,72356,Regular Structure (Мицелла),0.135,0.044,4.2,368.4,0.677
1,1,11420,Chaotic Motion (Хаос),0.133,0.072,3.5,396.6,0.808
2,2,74520,Sparse Distribution (Разреженное),0.133,0.042,3.6,413.1,0.666
3,3,23598,Sparse Distribution (Разреженное),0.135,-0.023,3.9,390.9,0.740


## 4. Multistep evaluation на fact-data

Проверяем качество autoregressive-прогноза: после seed-history `0..3` модель не использует фактические шаги, а прогнозирует сама себя вперед.

In [4]:
pipeline.load_predictor(train_if_missing=False)
multistep_predictions, multistep_metrics = pipeline.evaluate_multistep_on_fact_data(
    experiments,
    horizon=200,  # можно None для всего эксперимента
)
multistep_metrics

Using train_sample_ratio=0.350: 107/307 fact files
Selected demo experiment: 02_63_[45_bots_PWM_2_ex_158].pickle (test_files[2])


{'multistep_rmse_x': 171.2469830350214,
 'multistep_rmse_y': 169.6885114294418,
 'multistep_rmse_angle': 58.1643967030455,
 'multistep_ade_pos': 193.25324079498944,
 'multistep_fde_pos': 270.78168794174036,
 'multistep_final_horizon': 200,
 'total_predictions': 2763000}

In [5]:
multistep_predictions.head()

,file_name,slice_id,forecast_horizon,bot_id,coord_x_real,coord_y_real,angle_real,coord_x_pred,coord_y_pred,angle_pred
0,00_61_[45_bots_PWM_2_ex_175].pickle,4,1,2,1134.0,424.0,11.309933,1134.377930,421.519684,11.290463
1,00_61_[45_bots_PWM_2_ex_175].pickle,4,1,4,798.0,606.0,150.255112,796.367126,605.135254,151.595352
2,00_61_[45_bots_PWM_2_ex_175].pickle,4,1,5,1211.0,482.0,11.309933,1211.339233,479.982269,14.062761
3,00_61_[45_bots_PWM_2_ex_175].pickle,4,1,7,942.0,633.0,240.255112,942.243958,630.539368,241.467590
4,00_61_[45_bots_PWM_2_ex_175].pickle,4,1,9,897.0,247.0,240.255112,897.402771,244.367615,239.474701


## 4.1. Видео multistep после evaluation

Эта ячейка не запускает rollout повторно. Она использует `multistep_predictions` из предыдущей ячейки или сохранённый parquet.


In [6]:
demo_file = pipeline.select_demo_file(experiments)
# demo_file

multistep_video_path = pipeline.save_multistep_video(
    predictions_df=multistep_predictions,
    file_name=demo_file,
)
multistep_video_path

Using train_sample_ratio=0.350: 107/307 fact files
Selected demo experiment: 02_63_[45_bots_PWM_2_ex_158].pickle (test_files[2])
Animation created for 200 frames and 45 robots: results\research\videos\multistep_real_vs_predicted_02_63__45_bots_PWM_2_ex_158__pickle.mp4


WindowsPath('results/research/videos/multistep_real_vs_predicted_02_63__45_bots_PWM_2_ex_158__pickle.mp4')

## 5. Генерация сетки синтетических начальных условий

Можно менять радиусы, скорости, режимы скоростей и число повторов.

In [ ]:
base_spec = InitialConditionSpec(
    experiment_id="base",
    n_robots=config.n_robots,
    history_len=config.start_step + 1,
    dt=config.dt,
    dish_diameter=config.dish_diameter,
    robot_diameter=config.robot_size,
    min_center_distance=config.min_center_distance,
    seed=config.random_state,
)

specs = InitialStateGenerator.make_sweep_specs(
    base_spec,
    # For a dish diameter of 1000 and robot diameter of 60, centers can use max radius 470.
    # These values represent dense / medium / sparse initial systems.
    radius_values=[250, 350, 470],
    
    # speed_values=None,  # None => sample continuous random speeds from speed_range
    # speed_range=(config.speed_min, config.speed_max),
    # n_speed_samples=config.n_speed_samples,  # currently 3 sampled speed variants
    # speed_seed=config.random_state,

    speed_values=[10.0, 20.0, 35.0],
    speed_range=None,
    n_speed_samples=None,
    speed_seed=None,

    velocity_modes=["random", "aligned", "rotational", "inward"],
    repeats=2,
)

print(f"Sampled speed_mean values: {sorted({round(s.speed_mean, 3) for s in specs})}")
print(f"Synthetic experiments: {len(specs)}")
specs[:3]


Sampled speed_mean values: [20.972, 29.349, 31.465]
Synthetic experiments: 72


[InitialConditionSpec(experiment_id='synthetic_r250_v20.97_random_rep0', n_robots=50, history_len=4, dt=0.1, position_mode='disk', center_x=0.0, center_y=0.0, dish_diameter=1000.0, radius=250.0, radius_std=60.0, domain_size=1000.0, n_position_clusters=3, cluster_std=80.0, robot_diameter=60.0, min_center_distance=None, boundary_margin=None, max_position_attempts=30000, velocity_mode='random', speed_mean=20.971960993801307, speed_std=5.0, heading_mean_deg=0.0, heading_noise_deg=30.0, acceleration_noise=1.0, angular_speed_mean=0.0, angular_speed_std=5.0, seed=42),
 InitialConditionSpec(experiment_id='synthetic_r250_v20.97_random_rep1', n_robots=50, history_len=4, dt=0.1, position_mode='disk', center_x=0.0, center_y=0.0, dish_diameter=1000.0, radius=250.0, radius_std=60.0, domain_size=1000.0, n_position_clusters=3, cluster_std=80.0, robot_diameter=60.0, min_center_distance=None, boundary_margin=None, max_position_attempts=30000, velocity_mode='random', speed_mean=20.971960993801307, speed_

## 6. Synthetic study на 600 шагов

Для каждого синтетического эксперимента:

1. генерируется initial history длины `start_step+1`;
2. запускается multistep rollout на 600 шагов;
3. каждый шаг классифицируется Macrostate KMeans;
4. считается факт появления мицеллы, первый шаг и длительность.

In [ ]:
states_df, summary_df = pipeline.run_synthetic_study(
    specs,
    total_steps=config.synthetic_total_steps,
    save_trajectories=False,
)

summary_df.head()

In [ ]:
states_df.head()

## 6.1. Графики synthetic study для презентации

Создаются обзорные графики: вероятность мицеллы по начальным условиям, скорость формирования, длительность жизни мицеллы и распределение финальных состояний.

In [ ]:
synthetic_plot_paths = pipeline.create_synthetic_presentation_plots(summary_df)
synthetic_plot_paths

## 6.2. Выбор synthetic experiment по результату

Можно выбрать эксперимент, где мицелла появилась раньше всего, жила дольше всего, не появилась вовсе, или где итоговое/доминирующее состояние содержит нужный текст.

In [ ]:
first_micelle_exp = pipeline.find_synthetic_experiment("first_micelle", summary_df=summary_df)
longest_micelle_exp = pipeline.find_synthetic_experiment("longest_micelle", summary_df=summary_df)
# no_micelle_exp = pipeline.find_synthetic_experiment("no_micelle", summary_df=summary_df)
first_micelle_exp, longest_micelle_exp

## 6.3. Видео synthetic rollout с macrostate подписью

Видео строится для конкретного synthetic experiment. В заголовке каждого кадра отображаются step, cluster, state_type и micelle flag.

In [ ]:
synthetic_video_path = pipeline.save_synthetic_video(
    experiment_id=first_micelle_exp,
    states_df=states_df,
)
synthetic_video_path

## 7. Полностью автоматический запуск одной командой

Эта ячейка делает всё сразу: one-step train/reuse → KMeans → multistep evaluation → synthetic study.

Она дублирует этапы выше, поэтому обычно её запускают вместо отдельных ячеек.

In [ ]:
# result = pipeline.run_full_research(
#     specs,
#     experiments_dict=experiments,
#     train_one_step=True,
#     force_retrain=False,
#     evaluate_real_horizon=200,
#     synthetic_total_steps=600,
#     save_trajectories=False,
# )
# result["synthetic_summary"].head()

## Выходные файлы

Основные результаты сохраняются в:

```text
models/gnn_model_weights.pth
results/research/tables/one_step_training_history.csv
results/research/tables/one_step_test_metrics.csv
results/research/tables/fact_cluster_results.csv
results/research/tables/multistep_fact_metrics.csv
results/research/tables/synthetic_step_states.csv
results/research/tables/synthetic_summary.csv
results/research/plots/*.png
```